# 📝 의료 데이터 하이브리드 검색 과제 LV2(응용)

<img src="images/09_drugs_loaded_graph.png" width="1400">

교안 02의 흐름을 **단위프로젝트2_개편 버전 C의 e약은요 제품 문서**에 적용합니다.

**데이터**: 프로젝트 전체 519개 문서 중 과제용 14개 제품 문서·43개 청크를 사용합니다. 개체 270개·도메인 관계 459개에 원문·청크를 연결하면 그림의 327노드·516관계가 됩니다. Drug 45개 중 문서가 있는 제품은 14개이며 나머지는 상호작용 대상으로 언급된 약품입니다.

**풀이 방법**: 준비 셀부터 순서대로 실행하세요. 구분선 안의 `[작성]` 부분에서 핵심 코드를 완성합니다. 대입문은 변수명과 `= ...`만 제공하고 우변 전체를 직접 작성합니다. `...`는 호출식이나 여러 줄 코드로 바꿀 수 있습니다. 함수 틀·질문·반복 처리·결과 표와 출력은 제공됩니다. 앞 문항의 결과를 다음 문항에서 이어 쓰고, 자가채점 후에는 원문과 관계 근거도 읽으세요.

**관계 의미**: TREATS는 효능, HAS_SIDE_EFFECT는 이상반응, CAUTION_FOR는 주의 대상의 원문 기록입니다. 이 과제는 제품 문서 검색이며 복용 추천이 아닙니다. 같은 증상을 효능과 이상반응에 함께 언급할 수 있으므로 관계 종류를 구분합니다.

**모델 호출**: 한글 질문 기준 질문 임베딩 1회·Jev 언어 판별 1회·Cypher 생성 1회·답변 생성 1회입니다. 영어 질문일 때만 한국어 검색어 생성 1회가 추가됩니다. 문서 벡터는 배포 파일을 재사용합니다.


## 준비와 데이터 연결

README의 패키지·환경변수를 설정합니다. 영화 교안을 적재한 실습 DB를 그대로 사용할 수 있습니다. 아래 준비 코드는 의료 그래프를 생성·재사용하며, 검색 문제는 직접 작성합니다.


In [ ]:
# 검색·언어 판별·구조화 출력에 교안과 같은 라이브러리를 사용합니다.
import json
import os
import re
from pathlib import Path
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from pydantic import BaseModel, Field
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem

from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_classic.retrievers import EnsembleRetriever


In [ ]:
# 교안의 .env로 DB와 질문 임베딩 모델을 연결합니다.
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 질문을 자동으로 나누지 않습니다.
)

# 검색어·Cypher·답변에는 같은 모델을 사용합니다.
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


def run_cypher(query, **params):
    """파라미터 쿼리의 결과를 사전 목록으로 반환합니다."""
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


원문과 청크·벡터를 읽습니다. `drugs["relations"]`는 claim_id·subject_id·object_id·relation·evidence·source_doc_id·source_url을 가진 관계 목록입니다. `documents_by_id`는 원문 ID로 제목·본문·URL을, `nodes_by_id`는 개체 ID로 name·entity_type을 찾는 사전입니다.


In [ ]:
# 같은 프로젝트에서 검증한 문서·청크와 배포 벡터를 읽습니다.
import sys
sys.path.insert(0, str(material_dir.resolve()))
from graph_data import load_graph, read_vectors

drugs = load_graph(material_dir / "data/drugs_extraction_packet.json")
documents_by_id = drugs["documents"]
nodes_by_id = {node["standard_id"]: node for node in drugs["nodes"]}
chunk_ids, vectors = read_vectors(material_dir / "data/embeddings/drugs.npz", drugs["chunks"], embedding_model)
print("제품 문서:", len(documents_by_id), "청크:", len(chunk_ids))
display(pd.DataFrame([{"source_id": key, "title": doc["title"]} for key, doc in documents_by_id.items()]))


### 의료 그래프를 준비합니다

`DrugEntity`는 원본의 다섯 개체 유형에 덧붙인 공통 ID 레이블입니다. `DrugRankNode`는 이 과제의 Drug·Symptom에만 붙여 PageRank 계산 범위를 지정합니다. 원문은 `DrugDocument`, 청크는 `DrugChunk`에 적재합니다. 이 레이블과 `drug_chunks`·`drug_fulltext` 인덱스는 영화 교안의 검색 대상과 구분됩니다.


In [ ]:
# [제공 코드] 적재는 교안의 준비 단계를 재사용합니다. 검색 로직은 아래에서 직접 작성합니다.
# DrugEntity는 이 과제 개체의 공통 ID 제약을 위한 추가 레이블입니다.
run_cypher("CREATE CONSTRAINT drug_entity_id IF NOT EXISTS FOR (n:DrugEntity) REQUIRE n.standard_id IS UNIQUE")
run_cypher("CREATE CONSTRAINT drug_document_id IF NOT EXISTS FOR (n:DrugDocument) REQUIRE n.id IS UNIQUE")
run_cypher("CREATE CONSTRAINT drug_chunk_id IF NOT EXISTS FOR (n:DrugChunk) REQUIRE n.id IS UNIQUE")

# 원본 식별자는 파일에 보존하고 DB ID에는 과제 접두어를 붙입니다.
for entity_type in ["Drug", "Symptom", "RiskGroup", "Ingredient", "Manufacturer"]:
    rows = [node for node in drugs["nodes"] if node["entity_type"] == entity_type]
    rank_label = ":DrugRankNode" if entity_type in ["Drug", "Symptom"] else ""
    run_cypher(f"""
        UNWIND $rows AS item
        MERGE (n:DrugEntity {{standard_id: 'day49:' + item.standard_id}})
        SET n:{entity_type}{rank_label}, n.name = item.name, n.aliases = item.aliases
    """, rows=rows)

# 같은 claim_id를 재사용해 중복 적재를 막고 원문 근거를 보존합니다.
for relation_type in ["TREATS", "HAS_SIDE_EFFECT", "CAUTION_FOR", "CONTAINS", "INTERACTS_WITH", "MADE_BY"]:
    rows = [row for row in drugs["relations"] if row["relation"] == relation_type]
    run_cypher(f"""
        UNWIND $rows AS item
        MATCH (a:DrugEntity {{standard_id: 'day49:' + item.subject_id}})
        MATCH (b:DrugEntity {{standard_id: 'day49:' + item.object_id}})
        MERGE (a)-[r:{relation_type} {{claim_id: item.claim_id}}]->(b)
        SET r.evidence = item.evidence, r.source_url = item.source_url,
            r.source_doc_id = item.source_doc_id
    """, rows=rows)


In [ ]:
# [제공 코드] 원문은 문서 제품에만 연결합니다. 상호작용 대상으로 나온 약명에는 문서가 없습니다.
run_cypher("""
    UNWIND $rows AS item
    MERGE (d:DrugDocument {id: item.doc_id})
    SET d.title = item.title, d.text = item.text, d.url = item.url
    WITH d, item
    MATCH (p:DrugEntity:Drug {standard_id: 'day49:rag:drugs:drug:' + item.doc_id})
    MERGE (d)-[:ABOUT_DRUG]->(p)
""", rows=[{"doc_id": key, **doc} for key, doc in documents_by_id.items()])

# 검사한 벡터 행 순서에 맞춰 청크를 저장합니다.
rows = [{"id": chunk_ids[i], "source_id": chunk.metadata["source_doc_id"],
         "text": chunk.page_content, "embedding": vectors[i].tolist()}
        for i, chunk in enumerate(drugs["chunks"])]
run_cypher("""
    UNWIND $rows AS item
    MATCH (d:DrugDocument {id: item.source_id})
    MERGE (c:DrugChunk {id: item.id})
    SET c.text = item.text, c.source_id = item.source_id, c.embedding = item.embedding
    MERGE (c)-[:FROM_DOCUMENT]->(d)
""", rows=rows)

# 의료 전용 레이블을 색인해 교안의 영화 검색과 구분합니다.
run_cypher("""
    CREATE VECTOR INDEX drug_chunks IF NOT EXISTS FOR (c:DrugChunk) ON c.embedding
    OPTIONS {indexConfig: {`vector.dimensions`: 768, `vector.similarity_function`: 'cosine'}}
""")
run_cypher("""
    CREATE FULLTEXT INDEX drug_fulltext IF NOT EXISTS FOR (c:DrugChunk) ON EACH [c.text]
    OPTIONS {indexConfig: {`fulltext.analyzer`: 'standard-no-stop-words'}}
""")
run_cypher("CALL db.awaitIndex('drug_chunks', 300)")
run_cypher("CALL db.awaitIndex('drug_fulltext', 300)")
print("의료 그래프와 두 검색 인덱스 준비 완료")


## 1. 원문 요구와 관계 조건을 나누어 검색합니다

**배경**: 효능 설명은 문서로 검색하고 공통 증상·주의 대상은 그래프에서 확인합니다.

**제공 코드**: 검색 함수·Jev 언어 분기·구조화 검색어 체인·질문 분리·질문 임베딩·결과 표를 제공합니다.

**요구사항**:

- <strong><code>dense_rows</code></strong>에는 vector_documents(query_vector, top_k=20)의 결과를 담으세요. 준비된 질문 벡터를 재사용합니다.
- <strong><code>fulltext_rows</code></strong>에는 fulltext_documents(semantic_query, top_k=20)의 결과를 담으세요. 벡터 검색과 같은 검색어를 사용합니다.

**확인 기준**: 두 검색 결과가 원문별 한 행이며 관계 조건은 graph_request의 자연어 질문에 담겨 있습니다.


In [ ]:
def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # content는 청크 본문, metadata는 청크 ID·원문 ID·유사도를 담는 사전입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
vector_retriever = VectorRetriever(
    driver, index_name="drug_chunks", embedder=embedding_model,
    result_formatter=format_vector,
)


def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 제품별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    result = vector_retriever.search(query_vector=query_vector, top_k=top_k)
    # 제품별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(dict(row))
    return rows

# 한국어 원문이므로 Jev의 영어 판단일 때만 검색어를 한국어로 바꿉니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영문 약품명이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


class KoreanKeywords(BaseModel):
    """한국어 원문에서 찾을 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 한국어 핵심어 3~6개")


# 영어 문장에 있는 내용만 한국어 검색어로 변환합니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "영어로 작성된 의약품 문서 검색 질문을 한국어 핵심어 3~6개로 바꾸세요. "
               "각 항목에는 핵심어 하나만 담고 중복을 제거하세요. "
               "질문에 없는 제품명·성분·효능을 추측해 추가하지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(KoreanKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 한국어 전문 검색 후 원문별 최고 점수 청크를 반환합니다."""
    decision = language_classifier.invoke({"state": query, "questions": {"language": language_question}})
    language = decision.choices["language"].choice
    # 한글 입력은 그대로 검색하고 영어만 한국어 검색어로 변환합니다.
    if language == "en":
        terms = lexical_chain.invoke({"question": query})
        search_text = " OR ".join(f'"{word}"' for word in terms.keywords)
    else:
        search_text = query
    print("질문 언어:", language, "| 전문 검색어:", search_text)
    hits = run_cypher("""
        MATCH (node:DrugChunk)
        SEARCH node IN (FULLTEXT INDEX drug_fulltext FOR $search_text LIMIT $top_k) SCORE AS score
        RETURN node.source_id AS source_id, node.id AS chunk_id, score
        ORDER BY score DESC, chunk_id
    """, search_text=search_text, top_k=top_k)
    rows, seen = [], set()
    # 청크 수가 많은 제품이 문서 후보를 여러 번 차지하지 않게 합니다.
    for row in hits:
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(row)
    return rows

# 전체 질문과 두 검색 경로의 입력을 나눕니다.
question = "게보린정(수출명:돌로린정)과 효능 증상을 3개 이상 공유하고 임부의 주의 관계가 기록된 제품 중, 콧물 코막힘 재채기 기침 가래를 효능으로 설명하는 제품 문서를 찾아 주세요. 공통 효능 증상과 임부 주의 문구도 근거로 확인해 주세요."
semantic_query = "콧물 코막힘 재채기 기침 가래"
graph_request = "제품명 '게보린정(수출명:돌로린정)'과 효능 증상을 3개 이상 공유하고, 주의 대상 이름이 정확히 '임부'인 다른 제품 문서를 찾아 주세요. 기준 제품은 제외하고 공통 효능 증상과 양쪽 효능 관계, 후보의 주의 관계 ID도 함께 알려 주세요."
query_vector = embedding_model.embed_query(semantic_query)
# ====================================================================
# [작성]
dense_rows = ...
# ====================================================================
# ====================================================================
# [작성]
fulltext_rows = ...
# ====================================================================
display(pd.DataFrame(dense_rows))
display(pd.DataFrame(fulltext_rows))


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(language_classifier, TypeSafeClassifier), "Jev 언어 판별기를 생성하세요."
assert language_classifier.model == os.getenv("TYPESAFE_MODEL", "jev-1.13.0"), "Jev 모델 설정을 확인하세요."
assert isinstance(language_question, Choice) and set(language_question.criteria) == {"ko", "en"}, "ko·en 선택 기준을 확인하세요."
assert KoreanKeywords.model_fields["keywords"].annotation == list[str], "keywords는 문자열 목록입니다."
assert callable(fulltext_documents), "언어 판단과 전문 검색 함수를 작성하세요."

assert len(query_vector) == 768 and dense_rows and fulltext_rows, '1번 원문 요구와 관계 조건을 나누어 검색합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(len(rows) == len({row["source_id"] for row in rows}) for rows in [dense_rows, fulltext_rows]), '1번 원문 요구와 관계 조건을 나누어 검색합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 1번 핵심 확인 완료!")


## 2. GraphCypherQAChain으로 관계 후보를 찾습니다

**배경**: Text-to-Cypher로 자연어 조건을 Cypher로 바꾸고 Neo4j에서 실행해 후보와 근거를 받습니다.

**제공 코드**: DB 연결·프롬프트·실행 결과 및 생성 Cypher 추출·출력.

**요구사항**:

- <strong><code>graph_chain</code></strong>은 GraphCypherQAChain.from_llm으로 생성합니다. llm=llm·graph=graph·cypher_prompt=cypher_prompt를 연결하세요.
- <strong><code>graph_chain</code></strong>의 include_types는 Drug·Symptom·RiskGroup·DrugDocument·TREATS·CAUTION_FOR·ABOUT_DRUG 목록입니다. return_direct=True·return_intermediate_steps=True·top_k=100·allow_dangerous_requests=True를 지정하세요. 마지막 옵션은 실행 위험에 동의하는 설정이며 DB 권한을 제한하지 않습니다.
- <strong><code>graph_response</code></strong>에는 graph_chain.invoke에 {"query": graph_request}를 전달한 결과를 담으세요.

**확인 기준**: 생성 쿼리에 기준 제품 제외·효능 관계·공통 증상 수·주의 대상 조건이 반영됩니다.


In [ ]:
# 생성 Cypher용 연결은 교안과 같은 실습 DB 계정을 사용합니다.
graph = Neo4jGraph(
    url=neo4j_uri, username=os.environ["NEO4J_USER"],
    password=os.environ["NEO4J_PASSWORD"], enhanced_schema=False,
)
# 체인이 {schema}에는 DB에서 읽어 include_types로 추린 구조를,
# {question}에는 invoke에 전달한 질문을 자동으로 넣습니다.
cypher_prompt = PromptTemplate.from_template("""
다음 스키마와 질문만 사용해 조회용 Cypher 하나를 작성하세요.
{schema}
관계 구조와 변수의 의미는 다음과 같습니다.
(anchor:Drug)-[first:TREATS]->(symptom:Symptom)<-[second:TREATS]-(candidate:Drug)
(d:DrugDocument)-[:ABOUT_DRUG]->(candidate)
(candidate)-[caution:CAUTION_FOR]->(risk:RiskGroup)
anchor는 기준 제품, candidate는 검색할 후보, d는 후보 문서입니다.
TREATS는 효능이며 HAS_SIDE_EFFECT는 이상반응입니다. CAUTION_FOR는 주의 기록입니다.
질문의 제품명과 위험군 이름은 name 속성에 정확히 대응합니다. 괄호·수출명을 포함하고 한국어 조사는 제외하세요.
제품 이름·공통 증상 수·주의 대상·기준 제품 제외 여부 등 검색 조건은 질문에서 해석하세요.
원문 유사도를 그래프 속성으로 추측하지 마세요.
아래 조회 구조를 사용하되 꺾쇠괄호 표시 세 곳을 질문에서 읽은 값으로 바꾸세요.
제품명·위험군명은 문자열 값, 최소 공통 증상 수는 정수로 넣습니다.
MATCH (anchor:Drug)-[first:TREATS]->(symptom:Symptom)<-[second:TREATS]-(candidate:Drug)
MATCH (d:DrugDocument)-[:ABOUT_DRUG]->(candidate)
MATCH (candidate)-[caution:CAUTION_FOR]->(risk:RiskGroup)
WHERE anchor.name = '<기준 제품명>' AND risk.name = '<위험군명>' AND candidate <> anchor
WITH d.id AS source_id, d.title AS title, candidate.standard_id AS drug_id,
     count(DISTINCT symptom) AS shared_count,
     collect(DISTINCT {{symptom: symptom.name, first_claim: first.claim_id, second_claim: second.claim_id}}) AS paths,
     collect(DISTINCT caution.claim_id) AS caution_claims
WHERE shared_count >= <최소 공통 증상 수>
RETURN source_id, title, drug_id, shared_count, paths, caution_claims
ORDER BY source_id LIMIT 100
개체 조건은 WITH 앞 WHERE에서만 검사합니다. WITH 뒤에는 집계한 shared_count 조건만 둡니다.
WITH 뒤에는 anchor·candidate·first·second·caution 변수가 없으므로 다시 참조하지 마세요.
count·collect는 WITH에서만 사용하며 WHERE나 NONE 내부에는 넣지 마세요.
Cypher만 반환하세요.
질문: {question}
""")
# ====================================================================
# [작성]
graph_chain = ...
# ====================================================================

# 생성한 조회문과 실제 결과를 함께 확인합니다.
# ====================================================================
# [작성]
graph_response = ...
# ====================================================================
graph_rows = graph_response["result"]
generated_cypher = graph_response["intermediate_steps"][0]["query"]
print(generated_cypher)
display(pd.DataFrame(graph_rows))


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(graph_chain, GraphCypherQAChain), '2번 GraphCypherQAChain으로 관계 후보를 찾습니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert generated_cypher.strip() and graph_rows, '2번 GraphCypherQAChain으로 관계 후보를 찾습니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all({"source_id", "title"} <= row.keys() for row in graph_rows), '2번 GraphCypherQAChain으로 관계 후보를 찾습니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 2번 핵심 확인 완료!")


## 3. 생성 쿼리의 후보와 관계 근거를 확인합니다

**배경**: Text-to-Cypher가 반환한 후보를 후속 검색과 원문 근거에 연결합니다.

**제공 코드**: 원본 관계 사전 생성·후보 표·실제 관계 근거 출력 반복문을 제공합니다.

**요구사항**:

- <strong><code>allowed_ids</code></strong>에 graph_rows의 source_id 집합을 담으세요.
- <strong><code>returned_claim_ids</code></strong>에 graph_rows 각 행의 paths에 있는 first_claim·second_claim과 caution_claims를 중복 없는 집합으로 합치세요.
- <strong><code>unknown_claim_ids</code></strong>에 원본 관계 사전에 없는 ID만 남기세요.

**확인 기준**: 후보 ID가 중복되지 않고 반환 필드와 원본 관계 ID를 확인합니다. 이 구조 확인만으로 질문 해석이 맞았다고 보장하지 않습니다.


In [ ]:
# Text-to-Cypher가 반환한 후보 ID를 후속 검색에서 사용합니다.
# ====================================================================
# [작성]
allowed_ids = ...
# ====================================================================
relations_by_id = {row["claim_id"]: row for row in drugs["relations"]}
# ====================================================================
# [작성]
returned_claim_ids = ...
# ====================================================================
# ====================================================================
# [작성]
unknown_claim_ids = ...
# ====================================================================
display(pd.DataFrame(graph_rows))
print("원본에 없는 관계 ID:", unknown_claim_ids)
for claim_id in sorted(returned_claim_ids - unknown_claim_ids):
    claim = relations_by_id[claim_id]
    print(claim_id, claim["relation"], claim["evidence"], claim["source_url"])


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert allowed_ids == {row["source_id"] for row in graph_rows}, '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert len(graph_rows) == len(allowed_ids), '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert [row["source_id"] for row in graph_rows] == sorted(allowed_ids), '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all({"source_id", "title", "drug_id", "shared_count", "paths", "caution_claims"} <= row.keys() for row in graph_rows), '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(row["source_id"] in documents_by_id and isinstance(row["paths"], list) and isinstance(row["caution_claims"], list) for row in graph_rows), '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all({"symptom", "first_claim", "second_claim"} <= path.keys() for row in graph_rows for path in row["paths"]), '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(row["shared_count"] == len({path["symptom"] for path in row["paths"]}) for row in graph_rows), '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(len(row["caution_claims"]) == len(set(row["caution_claims"])) for row in graph_rows), '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert relations_by_id == {row["claim_id"]: row for row in drugs["relations"]}, '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert returned_claim_ids == {path[key] for row in graph_rows for path in row["paths"] for key in ["first_claim", "second_claim"]} | {claim_id for row in graph_rows for claim_id in row["caution_claims"]}, '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert unknown_claim_ids == returned_claim_ids - set(relations_by_id), '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert not unknown_claim_ids, '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(relations_by_id[path[key]]["relation"] == "TREATS" for row in graph_rows for path in row["paths"] for key in ["first_claim", "second_claim"]), '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(relations_by_id[claim_id]["relation"] == "CAUTION_FOR" for row in graph_rows for claim_id in row["caution_claims"]), '3번 생성 쿼리의 후보와 관계 근거를 확인합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 3번 핵심 확인 완료!")


## 4. EnsembleRetriever로 전문·벡터 검색을 합칩니다

**배경**: 서로 다른 두 검색 순위를 LangChain의 RRF 구현으로 통합합니다.

**제공 코드**: as_documents·RunnableLambda 검색기·Document 변환·검색 순위 및 역수 점수 구성·표를 제공합니다.

**요구사항**:

- <strong><code>ensemble_retriever</code></strong>는 EnsembleRetriever입니다. retrievers=[fulltext_retriever, dense_retriever], weights=[0.4, 0.6], c=60, id_key="source_id"로 생성하세요.
- <strong><code>fused_docs</code></strong>에는 ensemble_retriever.weighted_reciprocal_rank에 [fulltext_docs, dense_docs]를 전달한 결과를 담으세요. 이미 받은 목록을 사용하므로 재검색하지 않습니다.

**확인 기준**: 원문 ID로 중복을 제거한 통합 순서가 나오며 전문·벡터 검색을 다시 호출하지 않습니다.


In [ ]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_classic.retrievers import EnsembleRetriever


def as_documents(rows):
    """원문별 검색 결과를 LangChain Document 목록으로 바꿉니다."""
    return [Document(page_content=documents_by_id[row["source_id"]]["text"],
                     metadata={"source_id": row["source_id"]}) for row in rows]


# 기존 검색 함수를 LangChain에서 호출할 수 있도록 연결합니다.
# 두 함수는 이미 원문별 중복을 제거한 결과를 반환합니다.
fulltext_retriever = RunnableLambda(
    lambda query: as_documents(fulltext_documents(query, top_k=30))
)
dense_retriever = RunnableLambda(
    lambda query: as_documents(vector_documents(embedding_model.embed_query(query), top_k=30))
)
# 전문·벡터 검색만 가중 RRF로 결합합니다. 같은 원문은 source_id로 합칩니다.
# ====================================================================
# [작성]
ensemble_retriever = ...
# ====================================================================

# 이미 받은 두 검색 결과를 Document로 바꾸어 재검색 없이 합칩니다.
fulltext_docs = as_documents(fulltext_rows)
dense_docs = as_documents(dense_rows)
# ====================================================================
# [작성]
fused_docs = ...
# ====================================================================
# PageRank와 결합할 점수는 통합 결과의 순위 역수로 정의합니다.
fused_rows = [
    {"source_id": doc.metadata["source_id"], "search_rank": rank, "search_score": 1 / rank}
    for rank, doc in enumerate(fused_docs, 1)
]
display(pd.DataFrame(fused_rows))


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(ensemble_retriever, EnsembleRetriever), '4번 EnsembleRetriever로 전문·벡터 검색을 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert isinstance(fulltext_retriever, RunnableLambda) and isinstance(dense_retriever, RunnableLambda), '4번 EnsembleRetriever로 전문·벡터 검색을 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert ensemble_retriever.retrievers == [fulltext_retriever, dense_retriever], '4번 EnsembleRetriever로 전문·벡터 검색을 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert ensemble_retriever.weights == [0.4, 0.6] and ensemble_retriever.c == 60 and ensemble_retriever.id_key == "source_id", '4번 EnsembleRetriever로 전문·벡터 검색을 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(isinstance(doc, Document) and doc.page_content == documents_by_id[doc.metadata["source_id"]]["text"] and set(doc.metadata) == {"source_id"} for doc in fulltext_docs + dense_docs), '4번 EnsembleRetriever로 전문·벡터 검색을 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert [doc.metadata["source_id"] for doc in fulltext_docs] == [row["source_id"] for row in fulltext_rows], '4번 EnsembleRetriever로 전문·벡터 검색을 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert [doc.metadata["source_id"] for doc in dense_docs] == [row["source_id"] for row in dense_rows], '4번 EnsembleRetriever로 전문·벡터 검색을 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert fused_docs == ensemble_retriever.weighted_reciprocal_rank([fulltext_docs, dense_docs]), '4번 EnsembleRetriever로 전문·벡터 검색을 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert len(fused_rows) == len({row["source_id"] for row in fulltext_rows + dense_rows}), '4번 EnsembleRetriever로 전문·벡터 검색을 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert fused_rows == [{"source_id": doc.metadata["source_id"], "search_rank": rank, "search_score": 1 / rank} for rank, doc in enumerate(fused_docs, 1)], '4번 EnsembleRetriever로 전문·벡터 검색을 합칩니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 4번 핵심 확인 완료!")


## 5. 관계 조건으로 제한하고 검색 누락 후보를 보완합니다

**배경**: Text-to-Cypher 후보 안에서 검색 순서를 유지하고 문서 검색이 놓친 후보도 확인합니다.

**제공 코드**: 기존 후보·관계 후보 집합과 비교 출력 반복문을 제공합니다.

**요구사항**:

- <strong><code>text_candidates</code></strong>는 fused_rows의 원래 순서를 유지하며 source_id가 allowed_ids에 속한 행만 남긴 목록입니다.
- <strong><code>graph_added_ids</code></strong>는 관계 후보 중 fused_rows에 없는 원문 ID 집합입니다.
- <strong><code>candidates</code></strong>는 text_candidates 뒤에 누락 후보를 원문 ID순으로 붙인 새 목록입니다. 누락 후보는 graph_added_ids를 정렬해 만들며 각 행에 source_id·search_rank=None·search_score=0.0을 담으세요.

**확인 기준**: candidates의 ID 집합은 allowed_ids와 같고, 문서 검색에 없던 관계 후보는 뒤에 점수 0으로 추가됩니다.


In [ ]:
# 관계 조건 후보 안에서는 전문·벡터 통합 순서를 유지합니다.
# ====================================================================
# [작성]
text_candidates = ...
# ====================================================================
# ====================================================================
# [작성]
graph_added_ids = ...
# ====================================================================
# 문서 검색이 놓친 관계 후보는 검색 점수 0으로 뒤에 보완합니다.
# ====================================================================
# [작성]
candidates = ...
# ====================================================================
for label, rows in [("문서 검색 후보", text_candidates), ("관계 후보 보완", candidates)]:
    print(label, [documents_by_id[row["source_id"]]["title"] for row in rows])
print("관계 검색이 보완한 원문:", graph_added_ids)


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert text_candidates == [row for row in fused_rows if row["source_id"] in allowed_ids], '5번 관계 조건으로 제한하고 검색 누락 후보를 보완합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert graph_added_ids == allowed_ids - {row["source_id"] for row in fused_rows}, '5번 관계 조건으로 제한하고 검색 누락 후보를 보완합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert candidates == text_candidates + [{"source_id": source_id, "search_rank": None, "search_score": 0.0} for source_id in sorted(graph_added_ids)], '5번 관계 조건으로 제한하고 검색 누락 후보를 보완합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert len(candidates) == len(allowed_ids) and {row["source_id"] for row in candidates} == allowed_ids, '5번 관계 조건으로 제한하고 검색 누락 후보를 보완합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 5번 핵심 확인 완료!")


## 6. 효능 그래프의 PageRank를 계산합니다

**배경**: 제품·증상을 TREATS로 연결한 전체 그래프에서 제품의 중심성을 계산합니다.

**제공 코드**: 그래프 이름·이전 메모리 그래프 제거·제품 점수 조회·pagerank_by_id 생성·표를 제공합니다.

**요구사항**:

- <strong><code>gds.graph.project</code></strong>를 실행하는 run_cypher 호출문 전체를 첫 구간에 작성하세요. $name을 그래프 이름으로 사용하고 DrugRankNode와 TREATS를 투영하며 관계 설정은 {TREATS: {orientation: "UNDIRECTED"}}입니다. nodeCount·relationshipCount를 YIELD한 뒤 반환하세요. run_cypher에 name=projection_name을 전달합니다.
- <strong><code>gds.pageRank.write</code></strong>를 실행하는 run_cypher 호출문 전체를 두 번째 구간에 작성하세요. 그래프 이름은 $name이며 writeProperty="drug_pagerank"·dampingFactor=0.85입니다. nodePropertiesWritten을 YIELD한 뒤 반환하고 name=projection_name을 전달하세요.

**확인 기준**: 14개 원문마다 해당 제품의 양수 PageRank 점수가 있습니다.


In [ ]:
# 효능 관계의 구조만 비교하도록 Drug·Symptom과 TREATS를 투영합니다.
projection_name = "assignment_drug_symptoms"
run_cypher("CALL gds.graph.drop($name, false)", name=projection_name)
# ====================================================================
# [작성]
...
# ====================================================================
# ====================================================================
# [작성]
...
# ====================================================================
# 문서 제품 14개의 점수만 검색 결과와 같은 원문 ID로 연결합니다.
pagerank_rows = run_cypher("""
    MATCH (d:DrugDocument)-[:ABOUT_DRUG]->(p:Drug)
    RETURN d.id AS source_id, p.name AS title, p.drug_pagerank AS pagerank
    ORDER BY source_id
""")
pagerank_by_id = {row["source_id"]: row["pagerank"] for row in pagerank_rows}

display(pd.DataFrame(pagerank_rows))


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert len(pagerank_rows) == 14 and len(pagerank_by_id) == 14, '6번 효능 그래프의 PageRank를 계산합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert set(pagerank_by_id) == set(documents_by_id), '6번 효능 그래프의 PageRank를 계산합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(value > 0 for value in pagerank_by_id.values()), '6번 효능 그래프의 PageRank를 계산합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(pagerank_by_id[row["source_id"]] == row["pagerank"] for row in pagerank_rows), '6번 효능 그래프의 PageRank를 계산합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 6번 핵심 확인 완료!")


## 7. 같은 후보에서 PageRank 비중을 비교합니다

**배경**: 후보 집합을 고정한 상태에서 검색 관련성과 그래프 중요도를 결합합니다.

**제공 코드**: minmax·함수 정의·빈 목록 처리·후보 반복 및 정렬·세 비중 비교·원문 조회와 출력을 제공합니다. search_score가 RRF 내부 점수가 아닌 순위 역수임을 설명합니다.

**요구사항**:

- <strong><code>retrieval</code></strong>은 현재 rows의 search_score 목록을 minmax로 정규화한 값입니다. <strong><code>centrality</code></strong>는 같은 순서의 source_id로 pagerank_by_id에서 찾은 점수 목록을 minmax로 정규화한 값입니다.
- <strong><code>ranked</code></strong>에 현재 row의 모든 키를 복사한 사전을 추가하는 문장 전체를 작성하세요. search_norm=relevance, pagerank_norm=importance, final_score=(1-weight) × relevance + weight × importance를 함께 담습니다.

**확인 기준**: 세 비중에서 후보 ID는 같고, 비중 0에서는 순위 역수로 만든 검색 점수의 정규화 값이 최종 점수입니다.


In [ ]:
def minmax(values):
    """현재 후보 안에서 값을 0~1로 변환합니다. 동점만 있으면 모두 0.5입니다."""
    low, high = min(values), max(values)
    # 모두 동점이면 0.5로 통일합니다.
    return [0.5 if high == low else (value - low) / (high - low) for value in values]


def rerank_candidates(rows, weight):
    """같은 후보에 검색 순위 점수와 제품 PageRank를 결합합니다."""
    # 조건을 통과한 후보가 없으면 재정렬 없이 빈 목록을 반환합니다.
    if not rows:
        return []
    # 검색 순위 점수(1 / 순위)와 PageRank를 같은 후보 안에서 0~1로 바꿉니다.
    # ====================================================================
    # [작성]
    retrieval = ...
    # ====================================================================
    # ====================================================================
    # [작성]
    centrality = ...
    # ====================================================================
    ranked = []
    # 각 후보에 그 문서의 검색 점수와 PageRank 점수를 연결합니다.
    for row, relevance, importance in zip(rows, retrieval, centrality):
        # weight 비율만큼 PageRank를 반영해 새 점수를 만듭니다.
        # ====================================================================
        # [작성]
        ...
        # ====================================================================
    # 같은 후보를 결합 점수순으로 재정렬합니다.
    return sorted(ranked, key=lambda row: (-row["final_score"],
                                         row["search_rank"] or float("inf"), row["source_id"]))

# 재검색 없이 동일한 후보에 비중만 바꿉니다.
rankings = {weight: rerank_candidates(candidates, weight) for weight in [0, 0.25, 0.65]}
for weight, rows in rankings.items():
    print("PageRank 비중:", weight)
    display(pd.DataFrame([{**row, "title": documents_by_id[row["source_id"]]["title"]} for row in rows]))

# 비중별 첫 후보의 원문 ID를 중복 없이 모읍니다.
first_ids = list(dict.fromkeys(rows[0]["source_id"] for rows in rankings.values() if rows))
# 선택한 제품의 원문을 Neo4j에서 조회합니다.
first_documents = run_cypher("""
    UNWIND $source_ids AS source_id
    MATCH (d:DrugDocument {id: source_id})
    RETURN d.id AS source_id, d.title AS title, d.text AS text
""", source_ids=first_ids)
for doc in first_documents:
    print(doc["title"], doc["text"])


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert minmax([2, 2]) == [0.5, 0.5] and minmax([1, 3]) == [0.0, 1.0], '7번 같은 후보에서 PageRank 비중을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert minmax([2, 4, 10]) == [0.0, 0.25, 1.0], '7번 같은 후보에서 PageRank 비중을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(rows == sorted(rows, key=lambda row: (-row["final_score"], row["search_rank"] if row["search_rank"] is not None else float("inf"), row["source_id"])) for rows in rankings.values()), '7번 같은 후보에서 PageRank 비중을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(row["pagerank_norm"] == minmax([pagerank_by_id[candidate["source_id"]] for candidate in candidates])[[candidate["source_id"] for candidate in candidates].index(row["source_id"])] for rows in rankings.values() for row in rows), '7번 같은 후보에서 PageRank 비중을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert rerank_candidates([], 0.25) == [], '7번 같은 후보에서 PageRank 비중을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert set(rankings) == {0, 0.25, 0.65}, '7번 같은 후보에서 PageRank 비중을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all({row["source_id"] for row in rows} == {row["source_id"] for row in candidates} for rows in rankings.values()), '7번 같은 후보에서 PageRank 비중을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(0 <= row["search_norm"] <= 1 and 0 <= row["pagerank_norm"] <= 1 and abs(row["final_score"] - (1 - weight) * row["search_norm"] - weight * row["pagerank_norm"]) < 1e-12 for weight, rows in rankings.items() for row in rows), '7번 같은 후보에서 PageRank 비중을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(row["search_norm"] == minmax([candidate["search_score"] for candidate in candidates])[[candidate["source_id"] for candidate in candidates].index(row["source_id"])] for rows in rankings.values() for row in rows), '7번 같은 후보에서 PageRank 비중을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all("final_score" not in row for row in candidates), '7번 같은 후보에서 PageRank 비중을 비교합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 7번 핵심 확인 완료!")


**판단**: 세 비중의 첫 제품 원문을 읽고, 이 질문에 사용할 비중을 이유와 함께 적으세요. 첫 제품이 같다면 점수 차이를 비교하세요.


*(여기에 판단과 근거를 서술하세요)*


## 8. 멀티홉 결과를 원문과 관계 근거로 재구성합니다

**배경**: 제품 원문·공통 효능·주의 문구를 각각 실제 출처에 연결합니다.

**제공 코드**: 함수 정의·후보 및 원문 연결·근거 사전 구성·정렬·함수 호출·표를 제공합니다.

**요구사항**:

- <strong><code>claim_ids</code></strong> 집합에 현재 path의 first_claim·second_claim을 모두 추가하는 갱신문을 첫 구간에 작성하세요.
- <strong><code>claim_ids</code></strong>에 candidate["caution_claims"]의 ID도 모두 추가하는 갱신문을 두 번째 구간에 작성하세요. 중복은 집합에서 제거됩니다.

**확인 기준**: 후보 원문·공통 증상의 양쪽 근거·후보의 주의 관계가 빠짐없이 연결됩니다.


In [ ]:
# 양쪽 효능과 주의 관계의 ID로 원본 근거를 찾습니다.
relations_by_id = {row["claim_id"]: row for row in drugs["relations"]}


def build_evidence(ranked, graph_rows, top_n=3):
    """상위 제품의 원문·공통 효능·주의 문구 근거를 모읍니다."""
    graph_by_id = {row["source_id"]: row for row in graph_rows}
    evidence, claim_ids = [], set()
    for row in ranked[:top_n]:
        source_id = row["source_id"]
        doc = documents_by_id[source_id]
        candidate = graph_by_id[source_id]
        evidence.append({"evidence_id": source_id, "kind": "product_document", "title": doc["title"],
                         "url": doc["url"], "text": doc["text"]})
        # 공통 증상의 양쪽 관계와 후보의 주의 관계를 함께 보존합니다.
        for path in candidate["paths"]:
            # ====================================================================
            # [작성]
            ...
            # ====================================================================
        # ====================================================================
        # [작성]
        ...
        # ====================================================================
    for claim_id in sorted(claim_ids):
        claim = relations_by_id[claim_id]
        evidence.append({"evidence_id": claim_id,
                         "kind": "treats_relation" if claim["relation"] == "TREATS" else "caution_relation",
                         "title": nodes_by_id[claim["subject_id"]]["name"],
                         "target": nodes_by_id[claim["object_id"]]["name"],
                         "url": claim["source_url"], "text": claim["evidence"]})
    return evidence

# 답변에 제공할 후보와 근거를 고정합니다.
ranked = rankings[0.25]
evidence = build_evidence(ranked, graph_rows, top_n=3)
display(pd.DataFrame(evidence)[["evidence_id", "kind", "url"]])


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert evidence and len(evidence) == len({item["evidence_id"] for item in evidence}), '8번 멀티홉 결과를 원문과 관계 근거로 재구성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all({"evidence_id", "kind", "url", "text"} <= item.keys() and item["url"] and item["text"] for item in evidence), '8번 멀티홉 결과를 원문과 관계 근거로 재구성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert {item["evidence_id"] for item in evidence if item["kind"] == "product_document"} == {row["source_id"] for row in ranked[:3]}, '8번 멀티홉 결과를 원문과 관계 근거로 재구성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert {item["evidence_id"] for item in evidence if item["kind"] == "treats_relation"} == {path[key] for row in graph_rows if row["source_id"] in {r["source_id"] for r in ranked[:3]} for path in row["paths"] for key in ["first_claim", "second_claim"]}, '8번 멀티홉 결과를 원문과 관계 근거로 재구성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert {item["evidence_id"] for item in evidence if item["kind"] == "caution_relation"} == {claim_id for row in graph_rows if row["source_id"] in {r["source_id"] for r in ranked[:3]} for claim_id in row["caution_claims"]}, '8번 멀티홉 결과를 원문과 관계 근거로 재구성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(item["text"] == documents_by_id[item["evidence_id"]]["text"] and item["title"] == documents_by_id[item["evidence_id"]]["title"] and item["url"] == documents_by_id[item["evidence_id"]]["url"] for item in evidence if item["kind"] == "product_document"), '8번 멀티홉 결과를 원문과 관계 근거로 재구성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(item["text"] == relations_by_id[item["evidence_id"]]["evidence"] and item["url"] == relations_by_id[item["evidence_id"]]["source_url"] and item["title"] == nodes_by_id[relations_by_id[item["evidence_id"]]["subject_id"]]["name"] and item["target"] == nodes_by_id[relations_by_id[item["evidence_id"]]["object_id"]]["name"] for item in evidence if item["kind"] != "product_document"), '8번 멀티홉 결과를 원문과 관계 근거로 재구성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 8번 핵심 확인 완료!")


## 9. 근거를 인용하는 답변 체인을 작성합니다

**배경**: 실제로 검색한 자료만으로 효능과 주의 문구를 설명합니다.

**제공 코드**: 답변 프롬프트·GroundedAnswer 스키마·답변 출력을 제공합니다.

**요구사항**:

- <strong><code>answer_chain</code></strong>에는 answer_prompt와 llm.with_structured_output(GroundedAnswer)를 파이프 연산자로 연결한 체인을 담으세요.
- <strong><code>answer</code></strong>에는 answer_chain.invoke의 반환값을 담으세요. 입력 사전의 question에는 전체 question을, context에는 json.dumps(evidence, ensure_ascii=False)로 만든 문자열을 전달합니다.

**확인 기준**: 한국어 답변에 실제 원문과 관계 ID가 있고, 주의 문구의 조건과 의미가 유지됩니다.


In [ ]:
# 제품 문서의 효능과 주의 문구를 근거별로 확인하게 합니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공한 의약품 근거만 사용해 한국어 문서 검색 결과를 설명하세요. 다음 세 항목을 작성하세요. "
               "1. 질문의 원문 조건에 맞는 제품과 효능 설명. product_document를 인용하세요. "
               "2. 기준 제품과 선택 제품의 공통 효능 증상. 질문에서 요구한 개수 이상을 제시하세요. "
               "각 증상 뒤에는 target이 같고 title이 기준/선택 제품인 두 treats_relation의 ID를 붙이세요. "
               "표기 예: 공통 증상은 증상A[기준관계ID][후보관계ID], 증상B[기준관계ID][후보관계ID]입니다. "
               "3. 지정한 위험군의 주의 문구. 선택 제품의 caution_relation을 인용하세요. "
               "모든 인용은 [실제ID] 형식입니다. ID를 괄호나 쉼표 목록으로 쓰지 말고 각각 대괄호로 감싸세요. "
               "evidence_ids는 본문 대괄호 안에 실제로 쓴 ID만 중복 없이 담으세요. "
               "효능과 이상반응을 구분하고, 주의 문구의 복용 금지와 의사 또는 약사와 상의 지시를 구분하세요. "
               "복용을 추천하거나 PageRank를 약효·안전성 점수로 해석하지 마세요. 근거가 부족하면 확인할 수 없다고 답하세요."),
    ("human", "질문: {question}\n근거: {context}"),
])


class GroundedAnswer(BaseModel):
    """검색한 문서에 근거한 답변과 인용 ID 목록입니다."""
    answer: str = Field(description="제품 원문, 공통 효능 증상, 주의 문구의 세 항목. 본문에 ID별 대괄호 인용 포함")
    evidence_ids: list[str] = Field(description="본문의 모든 인용 ID를 중복 없이 담은 목록")


# ====================================================================
# [작성]
answer_chain = ...
# ====================================================================

# 실제 검색한 근거만 문맥으로 전달합니다.
# ====================================================================
# [작성]
answer = ...
# ====================================================================
print(answer.answer)


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(answer, GroundedAnswer) and answer.answer.strip(), '9번 근거를 인용하는 답변 체인을 작성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert re.search(r"[가-힣]", answer.answer), '9번 근거를 인용하는 답변 체인을 작성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert answer.evidence_ids and len(answer.evidence_ids) == len(set(answer.evidence_ids)), '9번 근거를 인용하는 답변 체인을 작성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert set(answer.evidence_ids) <= {item["evidence_id"] for item in evidence}, '9번 근거를 인용하는 답변 체인을 작성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(f"[{evidence_id}]" in answer.answer for evidence_id in answer.evidence_ids), '9번 근거를 인용하는 답변 체인을 작성합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 9번 핵심 확인 완료!")


## 10. 인용 원문과 답변의 주장을 대조합니다

**배경**: 인용 ID의 존재와 근거가 주장에 맞는지를 따로 확인합니다.

**제공 코드**: 근거 사전·본문 인용 정규식 추출·인용 근거 출력 반복문을 제공합니다.

**요구사항**:

- <strong><code>unknown_ids</code></strong>는 본문·목록 인용의 합집합에서 제공된 근거 ID를 뺀 집합입니다.
- <strong><code>citation_gap</code></strong>은 본문 인용과 목록 인용의 대칭차집합입니다.

**확인 기준**: unknown_ids·citation_gap이 비고 공통 효능·주의 근거가 실제 원문과 연결됩니다.


In [ ]:
# 인용 ID의 존재와 본문·목록의 일치를 확인합니다.
evidence_by_id = {item["evidence_id"]: item for item in evidence}
inline_ids = set(re.findall(r"\[([^\[\]]+)\]", answer.answer))
# ====================================================================
# [작성]
unknown_ids = ...
# ====================================================================
# ====================================================================
# [작성]
citation_gap = ...
# ====================================================================
print("미등록 ID:", unknown_ids, "본문과 목록 차이:", citation_gap)
for evidence_id in answer.evidence_ids:
    if evidence_id in evidence_by_id:
        item = evidence_by_id[evidence_id]
        print(evidence_id, item["kind"], item["title"], item.get("target", ""), item["url"], item["text"])


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert set(evidence_by_id) == {item["evidence_id"] for item in evidence}, '10번 인용 원문과 답변의 주장을 대조합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert inline_ids == set(re.findall(r"\[([^\[\]]+)\]", answer.answer)), '10번 인용 원문과 답변의 주장을 대조합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert unknown_ids == (inline_ids | set(answer.evidence_ids)) - set(evidence_by_id), '10번 인용 원문과 답변의 주장을 대조합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert citation_gap == inline_ids ^ set(answer.evidence_ids), '10번 인용 원문과 답변의 주장을 대조합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert not unknown_ids and not citation_gap, '10번 인용 원문과 답변의 주장을 대조합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert {evidence_by_id[key]["kind"] for key in answer.evidence_ids} == {"product_document", "treats_relation", "caution_relation"}, '10번 인용 원문과 답변의 주장을 대조합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert all(any(key in {path["first_claim"], path["second_claim"]} and {path["first_claim"], path["second_claim"]} <= set(answer.evidence_ids) for row in graph_rows for path in row["paths"]) for key in answer.evidence_ids if evidence_by_id[key]["kind"] == "treats_relation"), '10번 인용 원문과 답변의 주장을 대조합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
assert any(row["source_id"] in answer.evidence_ids and set(row["caution_claims"]) & set(answer.evidence_ids) and len({path["symptom"] for path in row["paths"] if {path["first_claim"], path["second_claim"]} <= set(answer.evidence_ids)}) > 0 for row in graph_rows), '10번 인용 원문과 답변의 주장을 대조합니다: 지문의 반환 키·값·중복·정렬 조건을 확인하세요.'
print("✅ 10번 핵심 확인 완료!")


**판단**: 선택 제품의 감기 증상 효능, 기준 제품과 공통 증상 3개 이상, 임부에 관한 실제 주의 문구를 각 근거 ID와 함께 설명하세요. 후보에 필요한 원문이 없을 때 PageRank 비중만 바꾸면 해결되는지도 적으세요.


*(여기에 판단과 근거를 서술하세요)*
